# View LSSTCam DeepCoadd Mosaic in Firefly

Display a mosaic of deep-coadd patches around a selected sky field using the
Firefly interactive viewer on the Rubin Science Platform.

**Workflow:**
1. Connect to the Butler repository and load the skymap.
2. Choose a target field (DDF or SV target) in the **Configuration** section.
3. Discover automatically which `deepCoadd` dataset type is available.
4. Identify the central tract/patch and its 3×3 neighbours.
5. Load all available patches for each LSST band.
6. Display the selected band in Firefly (one frame per patch).

- **Author:** Sylvie Dagoret-Campagne
- **Created:** 2026-03-26
- **Last update:** 2026-06-25
- **LSST pipelines:** w_2026_10


## Import

In [1]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.axes_grid1 import make_axes_locatable

from astropy.visualization import ImageNormalize, AsinhStretch, PercentileInterval, ZScaleInterval

from astropy.wcs import WCS
import matplotlib.pyplot as plt

import astropy.units as u


# import lsst.daf.butler as dafButler
from lsst.daf.butler import Butler

import lsst.geom as geom
from lsst.geom import SpherePoint, degrees
import lsst.afw.display as afwDisplay
from firefly_client import FireflyClient
import firefly_client.plot as ffplt

from lsst.skymap import PatchInfo, Index2D

In [29]:
!eups list -s

afw                   g41b6eb5643+ecf1fbc2c5 	current setup
alert_packet          g1362a8d7f1+5bdd4ba444 	current setup
analysis_ap           gbcb5411b99+ccb4eb47e6 	current setup
analysis_tools        gf0526598dc+3f52ab89b0 	current setup
ap_association        ga35a72f000+0ea0d63535 	current setup
ap_pipe               g67c9c00ab0+787026c3c7 	current setup
ap_verify             ga28814b29c+e869446758 	current setup
astro_metadata_translator gad30ce848d+3cbc7717b0 	current setup
astshim               g6483cd21eb+3aebfb091a 	current setup
atmospec              g1bc2adb031+d4de1ccac7 	current setup
base                  g53246c7159+663e2227e0 	current setup
cbp                   gdf01333386+28e30326ae 	current setup
cell_coadds           g0d63591232+937fdc30a3 	current setup
coadd_utils           g89139ef638+28e30326ae 	current setup
cp_pipe               g962ac060f5+c2f56f28fc 	current setup
cp_verify             gc28460b409+fb03421cd4 	current setup
cpputils              g87389fa792+b3

In [2]:
import gc

In [3]:
def dataset_type_exists(butler, name):
    try:
        butler.registry.getDatasetType(name)
        return True
    except KeyError:
        return False

In [4]:
def get_first_existing_dataset(butler, dataset_names, dataId, collections=None):
    for name in dataset_names:
        try:
            obj = butler.get(name, dataId, collections=collections)
            print(f"✔ Found dataset: {name}")
            return obj, name
        except Exception:
            continue

    raise ValueError("No valid dataset found among candidates.")

## Configuration

**Edit only this cell** to select the target field, the Butler collection,
and the band to display. Everything else runs automatically.


In [5]:
# ============================================================
# USER CONFIGURATION — edit only this cell to change the target
# ============================================================

# --- Butler repository and collections ---
#repo = "dp2_prep"
repo = "dp2"

collection = ["dp2"]

instrument = "LSSTCam"
skymapName = "lsst_cells_v2"

# --- Field selection ---
# Choose from the dictionaries defined in the next cell.
# Set field_dict to either lsstcam_targets or lsstcam_fields,
# then set key to the corresponding field name.
#
# lsstcam_targets keys : "47 Tuc", "Rubin SV 38 7", "Fornax dSph",
#                        "ECDFS", "EDFS", "Rubin SV 95 -25", "Seagull"
# lsstcam_fields  keys : "COSMOS", "XMM-LSS", "ELAIS-S1",
#                        "ECDFS", "EDFS_a", "EDFS_b"
#
USE_DDF = True  # True  → use lsstcam_fields (Deep Drilling Fields)
# False → use lsstcam_targets (SV / special targets)
key = "ECDFS"  # <-- change this to select any available field

# --- Band for the quick single-patch preview ---
BANDSEL = "r"  # preview band (used before the full mosaic loop)

# --- Band to display in Firefly ---
BAND_DISPLAY = "r"  # band to show in the Firefly mosaic at the end

In [6]:
# Derived configuration (do not edit)
where_clause = f"instrument = '{instrument}'"
collectionStr = collection[-1].replace("/", "_")

# Initialise the Butler repository
butler = Butler(repo, collections=collection)
registry = butler.registry
print(f"Butler initialised | repo: {repo}")
print(f"Collection : {collection[-1]}")

Butler initialised | repo: dp2
Collection : dp2


In [7]:
skymap = butler.get("skyMap", skymap=skymapName, collections=collection)

In [8]:
#camera = butler.get("camera", collections=collection, instrument=instrument)

## Discover available deepCoadd dataset type

The name of the deepCoadd dataset type varies between pipeline versions.
This cell probes the Butler registry and selects the first available name
from a prioritised list, so the notebook works with different collections.


In [9]:
# ----------------------------------------------------------------
# Automatic discovery of the available deepCoadd dataset type
# ----------------------------------------------------------------
# The dataset type name changed between pipeline versions:
#   Gen2  : 'deepCoadd_calexp'
#   DP1   : 'deep_coadd'
#   DP2+  : 'legacy_deep_coadd'
# We probe the registry instead of hardcoding the name so the
# notebook works across different Butler collections.

COADD_CANDIDATES = [
    "legacy_deep_coadd",
    "deep_coadd",
    "deepCoadd_calexp",
    "deepCoadd",
    "goodSeeingCoadd",
    "dcrCoadd",
    "deep_coadd_cell_predetection",
    "template_coadd",
]

# List all coadd-related dataset types actually present in the registry
all_coadd_types = [d.name for d in butler.registry.queryDatasetTypes() if "coadd" in d.name.lower()]
print("All coadd dataset types in registry:", all_coadd_types)

# Pick the first candidate that exists in the registry
COADD_DATASET = None
for name in COADD_CANDIDATES:
    if dataset_type_exists(butler, name):
        COADD_DATASET = name
        print(f"\n✔ Selected deepCoadd dataset type: '{COADD_DATASET}'")
        break

if COADD_DATASET is None:
    raise RuntimeError(
        "No recognised deepCoadd dataset type found in this Butler collection. "
        f"Available coadd types: {all_coadd_types}"
    )

All coadd dataset types in registry: ['deep_coadd_input_summary', 'deep_coadd', 'deepCoadd_dcr_ddec_consolidated_map_weighted_mean', 'deepCoadd_dcr_dra_consolidated_map_weighted_mean', 'deepCoadd_dcr_e1_consolidated_map_weighted_mean', 'deepCoadd_dcr_e2_consolidated_map_weighted_mean', 'deepCoadd_epoch_consolidated_map_max', 'deepCoadd_epoch_consolidated_map_mean', 'deepCoadd_epoch_consolidated_map_min', 'deepCoadd_exposure_time_consolidated_map_sum', 'deepCoadd_psf_e1_consolidated_map_weighted_mean', 'deepCoadd_psf_e2_consolidated_map_weighted_mean', 'deepCoadd_psf_maglim_consolidated_map_weighted_mean', 'deepCoadd_psf_size_consolidated_map_weighted_mean', 'deepCoadd_sky_background_consolidated_map_weighted_mean', 'deepCoadd_sky_noise_consolidated_map_weighted_mean']

✔ Selected deepCoadd dataset type: 'deep_coadd'


In [10]:
# print(camera.getName(),camera.getNameMap())

## List of Sky field of interest

In [11]:
lsstcam_targets = {}
lsstcam_targets["47 Tuc"] = {"field_name": "47 Tuc Globular Cluster", "ra": 6.02, "dec": -72.08}
lsstcam_targets["Rubin SV 38 7"] = {"field_name": "Low Ecliptic Latitude Field", "ra": 37.86, "dec": 6.98}
lsstcam_targets["Fornax dSph"] = {"field_name": "Fornax Dwarf Spheroidal Galaxy", "ra": 40.0, "dec": -34.45}
lsstcam_targets["ECDFS"] = {"field_name": "Extended Chandra Deep Field South", "ra": 53.13, "dec": -28.10}
lsstcam_targets["EDFS"] = {"field_name": "Euclid Deep Field South", "ra": 59.10, "dec": -48.73}
lsstcam_targets["Rubin SV 95 -25"] = {"field_name": "Low Galactic Latitude Field", "ra": 95.00, "dec": -25.0}
lsstcam_targets["Seagull"] = {"field_name": "Seagull Nebula", "ra": 106.23, "dec": -10.51}

In [12]:
lsstcam_fields = {}

# =========================
# Deep Drilling Fields (LSSTCam)
# =========================
lsstcam_fields["COSMOS"] = {"field_name": "COSMOS Deep Drilling Field", "ra": 150.11, "dec": 2.23}
lsstcam_fields["XMM-LSS"] = {"field_name": "XMM-LSS Deep Drilling Field", "ra": 35.57, "dec": -4.82}
lsstcam_fields["ELAIS-S1"] = {"field_name": "ELAIS-S1 Deep Drilling Field", "ra": 9.45, "dec": -44.02}
lsstcam_fields["ECDFS"] = {"field_name": "Extended Chandra Deep Field South", "ra": 52.98, "dec": -28.12}
# Euclid Deep Field South = 2 pointings LSSTCam
lsstcam_fields["EDFS_a"] = {"field_name": "Euclid Deep Field South (pointing a)", "ra": 58.90, "dec": -49.32}
lsstcam_fields["EDFS_b"] = {"field_name": "Euclid Deep Field South (pointing b)", "ra": 63.60, "dec": -47.60}

### Select the target

In [13]:
# Apply the field selection defined in the Configuration cell above
if USE_DDF:
    the_target = lsstcam_fields[key]
else:
    the_target = lsstcam_targets[key]

target_ra = the_target["ra"]
target_dec = the_target["dec"]
target_title = (
    the_target["field_name"] + f" | band {BANDSEL}" + f" | (RA, Dec) = ({target_ra:.2f}, {target_dec:.2f})"
)
target_point = SpherePoint(target_ra, target_dec, degrees)
print(f"Selected field : {key}")
print(f"  {target_title}")

Selected field : ECDFS
  Extended Chandra Deep Field South | band r | (RA, Dec) = (52.98, -28.12)


## Find the list of tract numbers from Object Table

In [14]:
datasettype = "object"
therefs = butler.registry.queryDatasets(datasettype, collections=collection)
tractsId_list = np.unique([ref.dataId["tract"] for ref in therefs])
tractsId_list = sorted(tractsId_list)
print(tractsId_list)

[np.int64(2054), np.int64(2055), np.int64(2056), np.int64(2057), np.int64(2058), np.int64(2078), np.int64(2079), np.int64(2080), np.int64(2081), np.int64(2173), np.int64(2174), np.int64(2175), np.int64(2208), np.int64(2209), np.int64(2210), np.int64(2211), np.int64(2212), np.int64(2213), np.int64(2232), np.int64(2233), np.int64(2234), np.int64(2235), np.int64(2236), np.int64(2237), np.int64(2330), np.int64(2331), np.int64(2332), np.int64(2333), np.int64(2334), np.int64(2367), np.int64(2368), np.int64(2369), np.int64(2370), np.int64(2371), np.int64(2372), np.int64(2392), np.int64(2393), np.int64(2394), np.int64(2395), np.int64(2396), np.int64(2397), np.int64(2398), np.int64(2492), np.int64(2493), np.int64(2494), np.int64(2495), np.int64(2496), np.int64(2497), np.int64(2532), np.int64(2533), np.int64(2534), np.int64(2535), np.int64(2536), np.int64(2537), np.int64(2557), np.int64(2558), np.int64(2559), np.int64(2560), np.int64(2561), np.int64(2562), np.int64(2660), np.int64(2661), np.int6

## Search all deepCoadd


In [15]:
# List all  deepCoadd_calexp which are in the butler collection
# Thus all patch and tracts
# refs = butler.registry.queryDatasets("deepCoadd_calexp", collections = collection)
# for ref in refs:
#    print(ref.dataId)

## Find the DataId

In [16]:
tract_info = skymap.findTract(target_point)
patch_info = tract_info.findPatch(target_point)
bbox = patch_info.getOuterBBox()

print("Patch bounding box:", bbox)

print("Tract ID :", tract_info.getId())
tractNbSel = tract_info.getId()

print("Patch Index :", patch_info.getIndex(), " , ", patch_info.getSequentialIndex())  # (x, y)
print("Bounding Box", bbox)

patchNbSel = patch_info.getSequentialIndex()

Patch bounding box: (minimum=(14850, 2850), maximum=(18149, 6149))
Tract ID : 5063
Patch Index : Index2D(x=5, y=1)  ,  15
Bounding Box (minimum=(14850, 2850), maximum=(18149, 6149))


In [17]:
central_patch = patch_info.getIndex()
central_x, central_y = patch_info.getIndex()
neighbor_patches = [
    f"{x},{y}"
    for x in range(central_x - 1, central_x + 2)
    for y in range(central_y - 1, central_y + 2)
    if 0 <= x <= 8 and 0 <= y <= 8
]

In [18]:
neighbor_patches_indexes = [
    Index2D(x=x, y=y)
    for x in range(central_x - 1, central_x + 2)
    for y in range(central_y - 1, central_y + 2)
    if 0 <= x <= 8 and 0 <= y <= 8
]

In [19]:
neighbor_patches_indexes

[Index2D(x=4, y=0),
 Index2D(x=4, y=1),
 Index2D(x=4, y=2),
 Index2D(x=5, y=0),
 Index2D(x=5, y=1),
 Index2D(x=5, y=2),
 Index2D(x=6, y=0),
 Index2D(x=6, y=1),
 Index2D(x=6, y=2)]

In [20]:
neighbor_patches_seqindexes = [
    tract_info[patch_index].getSequentialIndex() for patch_index in neighbor_patches_indexes
]

In [21]:
neighbor_patches_seqindexes

[4, 14, 24, 5, 15, 25, 6, 16, 26]

In [22]:
mapdict_patchesindexes = {}
for patch_index in neighbor_patches_indexes:
    patch_seqindex = tract_info[patch_index].getSequentialIndex()
    mapdict_patchesindexes[patch_seqindex] = f"{patch_index.x},{patch_index.y}"
mapdict_patchesindexes

{4: '4,0',
 14: '4,1',
 24: '4,2',
 5: '5,0',
 15: '5,1',
 25: '5,2',
 6: '6,0',
 16: '6,1',
 26: '6,2'}

In [23]:
# Add the patch and band to the dataId, we didn't need them for the objectTable_tract because it covers all patches and bands
# However the coadds are stored by patch and band dimensions so we have to add them to the dataId

dataId = {"band": BANDSEL, "tract": tractNbSel, "patch": patchNbSel, "skymap": skymapName}

In [24]:
print(f"dataId = {dataId}")

dataId = {'band': 'r', 'tract': 5063, 'patch': 15, 'skymap': 'lsst_cells_v2'}


## Fetch the DeepCoadd

In [25]:
# Fetch a single patch for a quick preview
# The dataset type name was discovered automatically above.
coadd_exp = butler.get(COADD_DATASET, dataId)

## Plot the (tract,patch) in matplotlib

In [26]:
image_array = coadd_exp.image.array
image = coadd_exp.image
wcs = coadd_exp.getWcs()
psf = coadd_exp.getPsf()

AttributeError: 'CellCoadd' object has no attribute 'getWcs'

In [28]:
coadd_exp.

<CellCoadd([y=2850:6150, x=14850:18150], tract=5063)>

In [ ]:
norm = ImageNormalize(
    image_array,
    interval=PercentileInterval(98.0),  # coupe les extrêmes
    stretch=AsinhStretch(),  # non-linéaire doux
)

fig, ax = plt.subplots(1, 1, figsize=(10, 10))
# im = ax.imshow(image_array, cmap="gray", norm=norm ,origin="lower", vmin=0, vmax=2000)
im = ax.imshow(image_array, cmap="gray", norm=norm, origin="lower")
ax.set_title(target_title)
plt.colorbar(im, ax=ax, pad=0.01, shrink=0.82, aspect=30)
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# WCS
# ─────────────────────────────────────────────────────────────
header = wcs.getFitsMetadata().toDict()
wcs_astropy = WCS(header)

# ─────────────────────────────────────────────────────────────
# Image scaling
# ─────────────────────────────────────────────────────────────
interval = ZScaleInterval()
vmin, vmax = interval.get_limits(image_array)

# ─────────────────────────────────────────────────────────────
# Figure (IMPORTANT: avoid constrained_layout with WCS + overlay)
# ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection=wcs_astropy)

# ─────────────────────────────────────────────────────────────
# Image
# ─────────────────────────────────────────────────────────────
im = ax.imshow(image_array, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)

ax.coords.grid(True, color="white", ls="dotted", alpha=0.5)

# ─────────────────────────────────────────────────────────────
# PRIMARY AXES (bottom / left)
# ─────────────────────────────────────────────────────────────
ra = ax.coords[0]
dec = ax.coords[1]

# RA bottom (HMS)
ra.set_axislabel("RA")
ra.set_format_unit(u.hourangle)
ra.set_major_formatter("hh:mm:ss")
ra.set_ticks_position("b")
ra.set_ticklabel_position("bottom")
ra.set_axislabel_position("bottom")

# Dec left (DMS)
dec.set_axislabel("Dec")
dec.set_format_unit(u.deg)
dec.set_major_formatter("dd:mm:ss")
dec.set_ticks_position("l")
dec.set_ticklabel_position("left")
dec.set_axislabel_position("left")


# ─────────────────────────────────────────────────────────────
# OVERLAY (top / right in degrees)
# ─────────────────────────────────────────────────────────────
overlay = ax.get_coords_overlay("icrs")

ra_o = overlay[0]
dec_o = overlay[1]

# RA top (deg)
ra_o.set_axislabel("RA [deg]")
ra_o.set_format_unit(u.deg)
ra_o.set_major_formatter("d.ddd")
ra_o.set_ticks_position("t")
ra_o.set_ticklabel_position("top")
ra_o.set_axislabel_position("top")

# Dec right (deg)
dec_o.set_axislabel("Dec [deg]")
dec_o.set_format_unit(u.deg)
dec_o.set_major_formatter("d.ddd")
dec_o.set_ticks_position("r")
dec_o.set_ticklabel_position("right")
dec_o.set_axislabel_position("right")

# IMPORTANT: only overlay ticks
ra_o.set_ticks_visible(True)
dec_o.set_ticks_visible(True)

# 🔥 IMPORTANT: disable unwanted sides explicitly
ax.tick_params(axis="x", top=False)
ax.tick_params(axis="y", right=False)
# disable top and right ticks for PRIMARY axes only
ax.tick_params(axis="x", bottom=True, top=False)
ax.tick_params(axis="y", left=True, right=False)


# ─────────────────────────────────────────────────────────────
# Astro convention
# ─────────────────────────────────────────────────────────────
ax.invert_xaxis()

# ─────────────────────────────────────────────────────────────
# Title (safe spacing)
# ─────────────────────────────────────────────────────────────
ax.set_title(f"DeepCoadd for {target_title}", pad=35)

# ─────────────────────────────────────────────────────────────
# Colorbar (clean + compact)
# ─────────────────────────────────────────────────────────────
cbar = plt.colorbar(im, ax=ax, pad=0.2, shrink=0.82, aspect=30)
cbar.set_label("Flux")

plt.show()

### Free memory

Release large objects that are no longer needed before loading the full mosaic.


In [ ]:
del image
del wcs
del psf
del coadd_exp
gc.collect()

## Check which patches have been generated from Object Table

In [ ]:
# keep a reference toward the objectTable_tract without loading it in memory
refs = butler.registry.queryDatasets("objectTable_tract", collections=collection)
for ref in refs:
    if ref.dataId["tract"] == tractNbSel:
        break

### use butler registry to access to patches

In [ ]:
patches = registry.queryDimensionRecords("patch", dataId={"tract": tractNbSel, "skymap": skymapName})

In [ ]:
listOfProcessedPatches = []
for patch_record in patches:
    # print(patch_record)
    if patch_record.id in neighbor_patches_seqindexes:
        listOfProcessedPatches.append(patch_record.id)
listOfProcessedPatches = sorted(listOfProcessedPatches)
listOfProcessedPatches = np.array(listOfProcessedPatches)

In [ ]:
print(listOfProcessedPatches)

- objectTable_tract is very big. Need to load one by one

In [ ]:
# Read ObjectTabe is memory death
# goodpatches = []

# for ipatch in neighbor_patches_seqindexes:
#    dataId_inTractPatch = {
#    "band": BANDSEL,
#    "tract": tractNbSel,
#    "patch": ipatch,
#    "skymap": skymapName
#    }

#    try:
#        object_table = butler.get("objectTable_tract", dataId_inTractPatch , collections=collection)
#        del object_table
#        gc.collect()
#        goodpatches.append(True)
#    except Exception as e:
#        print(f"Fails with patch {ipatch} : exception = {e}")
#        goodpatches.append(False)

In [ ]:
print(
    f"list of geometrically nearby patches from center patch {patchNbSel} in tract {tractNbSel} : ",
    neighbor_patches_seqindexes,
)
print(f"list of processed patches in tract {tractNbSel}", listOfProcessedPatches)

## Plot the Mosaic with Firefly per band

In [ ]:
afwDisplay.setDefaultBackend("firefly")
# display = afwDisplay.Display(frame=1)
# display.scale("asinh", "zscale")
# display.mtv(image, title = target_title)

In [ ]:
bands = ["u", "g", "r", "i", "z", "y"]

In [ ]:
all_deepCoaddsMosaics = {}
all_titles = {}

# Loop over all LSST bands
for idx, band in enumerate(bands):
    the_band = band

    the_title = key + f" band {band}"
    print(the_title)
    the_band_titles = []
    try:
        # Accumulate deepCoadds for this band to build the mosaic
        deepCoaddsMosaicSet = []
        # Loop over all neighbouring patches (sequential index)
        for ipatch in listOfProcessedPatches:
            print(ipatch)
            # Build the Butler dataId for this (band, tract, patch)
            the_dataId = {"band": the_band, "tract": tractNbSel, "patch": ipatch, "skymap": skymapName}

            current_title = the_title + f" tract = {tractNbSel} patch = {ipatch}"

            try:
                # Fetch the deepCoadd — dataset type was auto-discovered
                # Fetch the deepCoadd for this patch (dataset type auto-discovered)
                coadd_exp = butler.get(COADD_DATASET, the_dataId)
                # Add this exposure to the band mosaic set
                deepCoaddsMosaicSet.append(coadd_exp)
                the_band_titles.append(current_title)
            except Exception as e:
                print(f"Fails with patch {ipatch} : exception = {e}")
        nc = len(deepCoaddsMosaicSet)
        # Store the mosaic set for this band
        all_deepCoaddsMosaics[band] = deepCoaddsMosaicSet
        print(f"- added {nc} mosaics into all_deepCoaddsMosaics for band {band}")
        # Store the per-patch titles for this band
        all_titles[band] = the_band_titles

    except Exception as inst:
        print(f"{key} :: catch Exception for band {band}")
        print(type(inst))  # the exception type
        print(inst.args)  # arguments stored in .args
        print(inst)  # __str_

## Select band for Firefly display

Choose the band to visualise in the Firefly interactive viewer.
The `all_deepCoaddsMosaics` dictionary holds all loaded patches for
each band; only the selected band is sent to Firefly.


In [ ]:
# Use the band configured at the top of the notebook
band_sel = BAND_DISPLAY
deepCoaddsMosaics_sel = all_deepCoaddsMosaics[band_sel]
deepCoaddsMosaics_titles_sel = all_titles[band_sel]
print(f"Displaying band '{band_sel}' — {len(deepCoaddsMosaics_sel)} patches available")

In [ ]:
N = len(deepCoaddsMosaics_sel)
print(f"Number of coadds to plot : {N}")
for count in range(N):
    display = afwDisplay.Display(frame=count + 1)
    # cannot succeed to show white stars on dark sky
    # display.setImageColormap('gray')
    display.scale("asinh", "zscale")
    display.mtv(deepCoaddsMosaics_sel[count].image, title=deepCoaddsMosaics_titles_sel[count])

In [ ]:
# display.clearViewer()

In [ ]:
# setImageColormap) are “gray” and “grey”